# DeepJSCC, ADJSCC, and NTSCC_plus: consolidated 150-epoch comparison

This Kaggle notebook assembles the **completed** AWGN and Rayleigh training runs and their Kodak results. Run All regenerates the comparison tables, plots, and downloadable report CSVs from embedded copies of recorded outputs. It does not retrain the networks; each training run is linked below.

All three models completed 150 epochs per channel at 10 dB on DIV2K and were evaluated on all 24 Kodak images. DeepJSCC and ADJSCC each sampled 800 DIV2K images per epoch (128px crops). NTSCC_plus sampled 128 distinct images per epoch (256px crops), so its 150 epochs were not 150 full DIV2K passes.

The experiment named NTSCC++ uses the released author-linked **NTSCC_plus** implementation; it is not the unreleased online-editing NTSCC++ implementation. Rayleigh support for NTSCC_plus and ADJSCC is an extension of their author-release AWGN pipeline.

## Original training runs

- [DeepJSCC — 150 epochs/channel, T4 ×2](https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep)
- [ADJSCC — 150 epochs/channel](https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016)
- [NTSCC_plus — 150 epochs/channel, T4 ×2](https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016)
- [DeepJSCC FLOP profiler](https://www.kaggle.com/code/hmmkk20/deepjscc-flops-profile-20261017)

## Interpretation limits

- The analog models use nominal CBR 1/6; NTSCC_plus uses about 0.047–0.049. Quality is not rate-matched.
- FLOP counts are profiler estimates. Profile sizes differ (128px crop, 256px crop, or full Kodak image), optimizer updates are excluded, and fused operators may be under-counted.
- This notebook embeds summary and detailed metrics for all runs, full available DeepJSCC and NTSCC_plus epoch histories, and available per-image Kodak rows. ADJSCC per-image and epoch CSVs remain visible in its linked Kaggle output.


In [ ]:
import io, json, math, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Embedded artifacts are snapshots downloaded from the completed Kaggle runs.
SUMMARY_CSV = 'model,channel,snr_db,epochs,kodak_psnr_db,kodak_ssim,kodak_msssim,mean_complex_channel_uses_per_image,mean_cbr\nDeepJSCC,AWGN,10.0,150,23.368306047249074,0.6450856924057007,0.8727623720963796,196608.0,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,150,20.33186242830112,0.5323714216550192,0.7446102251609167,196608.0,0.16666666666666666\nADJSCC,AWGN,10.0,150,32.225783348083496,0.9200940628846487,0.9748047590255737,196608.0,0.16666666666666666\nADJSCC,Rayleigh,10.0,150,29.80873688062032,0.8685299406449,0.9542806545893351,196608.0,0.16666666666666666\nNTSCC_plus,AWGN,10.0,150,21.126831769943237,0.5454493500292301,0.7897162660956383,55442.333333333336,0.04699904830367477\nNTSCC_plus,Rayleigh,10.0,150,21.00322651863098,0.5700619394580523,0.7956210548679034,57522.333333333336,0.048762286150896984\n'
DETAIL_CSV = 'model,channel,snr_db,epochs,kodak_psnr_db,kodak_ssim,kodak_msssim,mean_complex_channel_uses_per_image,mean_cbr,div2k_pool_images,div2k_images_sampled_per_epoch,crop_px,batch_size,mixed_precision,training_flops_per_sample,training_flops_profile_input,inference_flops_per_image,inference_flops_profile_input,kaggle_kernel\nDeepJSCC,AWGN,10.0,150,23.368306047249074,0.6450856924057007,0.8727623720963796,196608.0,0.16666666666666666,800,800,128,32,false,167346181,one 128x128 RGB sample (forward + backward),167182338,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep\nDeepJSCC,Rayleigh,10.0,150,20.33186242830112,0.5323714216550192,0.7446102251609167,196608.0,0.16666666666666666,800,800,128,32,false,167362567,one 128x128 RGB sample (forward + backward),167182340,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep\nADJSCC,AWGN,10.0,150,32.225783348083496,0.9200940628846487,0.9748047590255737,196608.0,0.16666666666666666,800,800,128,16,false,70469104100,one 128x128 RGB sample (forward + backward; Adam update excluded),23490280974,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016\nADJSCC,Rayleigh,10.0,150,29.80873688062032,0.8685299406449,0.9542806545893351,196608.0,0.16666666666666666,800,800,128,16,false,70469112292,one 128x128 RGB sample (forward + backward; Adam update excluded),23490289166,one 128x128 RGB sample,https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016\nNTSCC++ (NTSCC_plus release),AWGN,10.0,150,21.126831769943237,0.5454493500292301,0.7897162660956383,55442.333333333336,0.04699904830367477,800,128,256,8,true (FP16 autocast),225518008470,one 256x256 RGB crop (PyTorch profiler; fused NATTEN may be under-counted),481300316170,one full-resolution 768x512 Kodak image (PyTorch profiler; fused NATTEN may be under-counted),https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016\nNTSCC++ (NTSCC_plus release),Rayleigh,10.0,150,21.00322651863098,0.5700619394580523,0.7956210548679034,57522.333333333336,0.048762286150896984,800,128,256,8,true (FP16 autocast),225518090394,one 256x256 RGB crop (PyTorch profiler; fused NATTEN may be under-counted),481300316174,one full-resolution 768x512 Kodak image (PyTorch profiler; fused NATTEN may be under-counted),https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016\n'
summary = pd.read_csv(io.StringIO(SUMMARY_CSV))
detail = pd.read_csv(io.StringIO(DETAIL_CSV))
assert len(summary) == 6 and set(summary.epochs.astype(int)) == {150}
assert set(summary.channel) == {"AWGN", "Rayleigh"}
display(Markdown("## Completed runs: 150 epochs per model and channel; 24 Kodak images per run"))
display(summary)


In [ ]:
# Quality metrics. Rate differs between NTSCC_plus and the two analog baselines.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric, title in zip(axes, ["kodak_psnr_db", "kodak_ssim", "kodak_msssim"], ["PSNR (dB)", "SSIM", "MS-SSIM"]):
    pivot = summary.pivot(index="model", columns="channel", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=18, color=["#377eb8", "#e41a1c"])
    ax.set_title(title); ax.set_xlabel(""); ax.grid(axis="y", alpha=.25); ax.legend(title="Channel")
fig.tight_layout(); plt.show()

display(Markdown("## Detailed compute and channel accounting"))
display(detail[["model","channel","epochs","training_flops_per_sample","training_flops_profile_input","inference_flops_per_image","inference_flops_profile_input","mean_complex_channel_uses_per_image","mean_cbr"]])


In [ ]:
# FLOPs are profiler estimates. Profile dimensions and operator coverage differ by implementation.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in zip(axes, ["training_flops_per_sample", "inference_flops_per_image"], ["Training FLOPs/sample", "Inference FLOPs/image"]):
    vals = detail.copy(); vals[metric] = pd.to_numeric(vals[metric], errors="coerce")
    vals["series"] = vals["model"] + " / " + vals["channel"]
    ax.barh(vals["series"], vals[metric], color=["#4daf4a" if c=="AWGN" else "#984ea3" for c in vals.channel])
    ax.set_xscale("log"); ax.set_title(title); ax.grid(axis="x", alpha=.25)
fig.tight_layout(); plt.show()


In [ ]:
# Epoch histories: full CSV records are embedded for DeepJSCC and NTSCC_plus.
EPOCH_CSVS = {'DeepJSCC_AWGN': 'model,channel,snr_db,epoch,train_mse,epoch_seconds,c,nominal_cbr\nDeepJSCC,AWGN,10.0,1,0.09982317596673966,4.197293519973755,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,2,0.06375875070691109,1.2483015060424805,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,3,0.03912618324160576,1.2486529350280762,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,4,0.03003191575407982,1.2438199520111084,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,5,0.025663421526551246,1.2595458030700684,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,6,0.02328482925891876,1.3676705360412598,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,7,0.021647571623325347,1.240717887878418,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,8,0.019484945088624955,1.241450309753418,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,9,0.018165112882852556,1.2474720478057861,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,10,0.01666639231145382,1.2651138305664062,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,11,0.016413725800812243,1.294379472732544,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,12,0.016003673635423182,1.2308876514434814,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,13,0.015007449239492416,1.2432050704956055,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,14,0.0145833295956254,1.38509202003479,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,15,0.01415074646472931,1.2371642589569092,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,16,0.01343945525586605,1.2746000289916992,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,17,0.01287841983139515,1.254958152770996,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,18,0.013399349227547645,1.2428641319274902,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,19,0.012495021298527718,1.2868523597717285,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,20,0.012619686163961887,1.2870957851409912,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,21,0.012144133262336255,1.2532305717468262,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,22,0.011913248337805271,1.39028000831604,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,23,0.011998238861560822,1.2947683334350586,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,24,0.011639730967581273,1.2684316635131836,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,25,0.011416011452674867,1.2873611450195312,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,26,0.011500311754643918,1.2781400680541992,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,27,0.011475364379584789,1.2463982105255127,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,28,0.01107743714004755,1.267517328262329,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,29,0.010988340433686971,1.3394780158996582,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,30,0.011021783500909805,1.3100380897521973,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,31,0.01080432053655386,1.278719186782837,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,32,0.010530222337692977,1.266674518585205,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,33,0.010336952023208142,1.2644257545471191,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,34,0.010328033044934272,1.3265788555145264,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,35,0.00999542374163866,1.249962329864502,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,36,0.010263652093708515,1.3006598949432373,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,37,0.010204471498727798,1.399427890777588,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,38,0.010429442543536425,1.3479549884796143,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,39,0.010216240808367728,1.286186933517456,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,40,0.010018894560635089,1.2691676616668701,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,41,0.009977224431931973,1.2775771617889404,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,42,0.009743478372693062,1.2893404960632324,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,43,0.009475208856165408,1.3251430988311768,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,44,0.009837986193597317,1.2866928577423096,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,45,0.00983631808310747,1.4124252796173096,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,46,0.009765866100788117,1.2834973335266113,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,47,0.00961010968312621,1.2574851512908936,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,48,0.009458941631019116,1.2946953773498535,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,49,0.009197963401675224,1.2906019687652588,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,50,0.009151438660919666,1.3117551803588867,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,51,0.008925956636667252,1.303335428237915,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,52,0.009150139093399047,1.3583059310913086,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,53,0.009193448033183813,1.2942049503326416,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,54,0.008770993575453759,1.278254508972168,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,55,0.00892398588359356,1.2999489307403564,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,56,0.009070281684398652,1.3069357872009277,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,57,0.008906671702861786,1.3163034915924072,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,58,0.00899079605937004,1.3628363609313965,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,59,0.008858937583863736,1.3623781204223633,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,60,0.00893871819600463,1.4616565704345703,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,61,0.008423540126532316,1.379606008529663,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,62,0.008555956911295652,1.39613938331604,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,63,0.008716745972633362,1.2985515594482422,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,64,0.008741822596639394,1.2910737991333008,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,65,0.008493680190294981,1.2930712699890137,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,66,0.008196206167340278,1.341245174407959,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,67,0.008119294438511133,1.4606339931488037,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,68,0.008359766229987145,1.3642446994781494,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,69,0.00828644772991538,1.34419584274292,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,70,0.008153857663273811,1.3322155475616455,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,71,0.008418982066214085,1.376147747039795,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,72,0.008263944666832685,1.3516874313354492,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,73,0.008471771869808435,1.38144850730896,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,74,0.008381890449672937,1.4625647068023682,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,75,0.008253294564783574,1.348433017730713,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,76,0.008343711905181407,1.3706138134002686,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,77,0.008208495862782002,1.3479597568511963,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,78,0.008444862477481364,1.3250653743743896,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,79,0.008127294853329659,1.363865852355957,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,80,0.007984977215528488,1.3521678447723389,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,81,0.008077950403094291,1.303823471069336,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,82,0.008186741042882204,1.3327536582946777,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,83,0.007946942094713449,1.3560853004455566,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,84,0.008383473586291075,1.3006889820098877,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,85,0.007934900652617215,1.3246047496795654,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,86,0.007949960548430682,1.3264949321746826,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,87,0.007977758925408125,1.3676364421844482,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,88,0.007794033382087946,1.4231922626495361,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,89,0.007707967925816774,1.4578447341918945,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,90,0.007712960634380579,1.4222297668457031,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,91,0.007955238055437804,1.4215939044952393,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,92,0.00775817833840847,1.3582863807678223,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,93,0.00801266822963953,1.3873860836029053,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,94,0.007615770921111107,1.3073632717132568,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,95,0.007753511071205139,1.3211603164672852,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,96,0.007864887956529856,1.4439818859100342,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,97,0.007600155100226403,1.3596558570861816,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,98,0.007594758570194244,1.3473353385925293,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,99,0.007706051636487246,1.3433876037597656,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,100,0.007553027756512165,1.3383145332336426,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,101,0.007854789607226848,1.417949914932251,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,102,0.0077864166162908074,1.3859453201293945,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,103,0.007718152906745672,1.4340848922729492,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,104,0.007514148093760014,1.3446428775787354,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,105,0.007404153030365706,1.3738934993743896,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,106,0.007357468791306019,1.334542989730835,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,107,0.007695039100944996,1.3988866806030273,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,108,0.007577766571193934,1.3338558673858643,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,109,0.007402731068432331,1.394545078277588,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,110,0.007264422252774239,1.3977515697479248,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,111,0.00744859829545021,1.468252420425415,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,112,0.007510757632553577,1.3463153839111328,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,113,0.007501307874917984,1.319075107574463,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,114,0.007319371812045574,1.3614006042480469,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,115,0.007479765210300684,1.3629536628723145,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,116,0.007761803381145,1.412583827972412,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,117,0.007343068718910217,1.363081693649292,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,118,0.007775384988635778,1.439662218093872,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,119,0.007296020314097405,1.3627588748931885,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,120,0.007647427096962929,1.4137251377105713,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,121,0.007414277959614992,1.3958065509796143,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,122,0.007397226002067328,1.3795077800750732,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,123,0.007430732827633619,1.345978021621704,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,124,0.007332437355071306,1.3604121208190918,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,125,0.0074461490660905835,1.481565237045288,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,126,0.007402000464498997,1.4142119884490967,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,127,0.00720812726765871,1.4079816341400146,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,128,0.007239011172205209,1.397003173828125,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,129,0.0070599007420241834,1.3957281112670898,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,130,0.0074168663285672665,1.4378268718719482,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,131,0.007290898989886045,1.404306411743164,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,132,0.00702288968488574,1.4992880821228027,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,133,0.00710004085674882,1.4262778759002686,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,134,0.006908147595822811,1.4234752655029297,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,135,0.00742160951718688,1.3998866081237793,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,136,0.007000453043729067,1.4671103954315186,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,137,0.007239385563880205,1.4343905448913574,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,138,0.0073544041812419894,1.451740026473999,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,139,0.007178279664367437,1.4962406158447266,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,140,0.0073311725258827206,1.4645228385925293,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,141,0.007100623659789562,1.4393725395202637,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,142,0.007237421479076147,1.4053130149841309,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,143,0.007006293628364801,1.419389247894287,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,144,0.007101038731634617,1.4187078475952148,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,145,0.007202534228563309,1.4441478252410889,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,146,0.00709148844704032,1.5003104209899902,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,147,0.00712133502587676,1.3766586780548096,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,148,0.00727725125849247,1.435020923614502,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,149,0.007260240036994219,1.464857816696167,8,0.16666666666666666\nDeepJSCC,AWGN,10.0,150,0.0072437833994627,1.446413278579712,8,0.16666666666666666\n', 'DeepJSCC_Rayleigh': 'model,channel,snr_db,epoch,train_mse,epoch_seconds,c,nominal_cbr\nDeepJSCC,Rayleigh,10.0,1,0.0988293132185936,1.597193717956543,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,2,0.0882253736257553,1.463083028793335,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,3,0.07116201475262642,1.4578368663787842,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,4,0.05853131726384163,1.4535129070281982,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,5,0.05075315684080124,1.445157766342163,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,6,0.04858627155423165,1.457226037979126,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,7,0.04673305854201317,1.4136710166931152,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,8,0.044232844933867456,1.5906472206115723,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,9,0.03865815915167332,1.4250385761260986,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,10,0.036325090304017066,1.4399840831756592,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,11,0.038596205040812494,1.4724342823028564,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,12,0.03582734256982803,1.4206819534301758,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,13,0.03718619398772716,1.462951421737671,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,14,0.03512484960258007,1.4924190044403076,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,15,0.03785569511353969,1.5184516906738281,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,16,0.03502297028899193,1.4803264141082764,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,17,0.03368144869804382,1.423351526260376,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,18,0.03386340901255608,1.4553024768829346,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,19,0.033019354790449144,1.4871439933776855,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,20,0.03300804637372494,1.4415359497070312,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,21,0.03335367746651172,1.5045230388641357,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,22,0.03202147789299488,1.4674978256225586,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,23,0.03484632126986981,1.4657583236694336,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,24,0.03417222328484058,1.4673707485198975,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,25,0.03414089776575566,1.490532636642456,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,26,0.030976130366325377,1.4845831394195557,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,27,0.028612087368965148,1.4844019412994385,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,28,0.02981865018606186,1.5844879150390625,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,29,0.03191207237541675,1.5371172428131104,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,30,0.030501200780272485,1.4770023822784424,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,31,0.029671077355742453,1.4826693534851074,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,32,0.02943895883858204,1.4557816982269287,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,33,0.026856513023376466,1.4267847537994385,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,34,0.027480105832219125,1.5391700267791748,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,35,0.02854961596429348,1.5536205768585205,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,36,0.029531653448939325,1.4514806270599365,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,37,0.027866246551275252,1.4345736503601074,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,38,0.02757535398006439,1.4752345085144043,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,39,0.027908382937312126,1.4671821594238281,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,40,0.027402280643582345,1.4954724311828613,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,41,0.028959313929080962,1.5561540126800537,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,42,0.026062045767903327,1.5009219646453857,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,43,0.025406382977962494,1.513467788696289,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,44,0.025177910067141054,1.49113130569458,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,45,0.029293771535158157,1.465512752532959,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,46,0.029257982671260833,1.5031514167785645,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,47,0.03038719519972801,1.510906457901001,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,48,0.027292762771248816,1.5649425983428955,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,49,0.026416086554527284,1.5262961387634277,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,50,0.02573420315980911,1.4904747009277344,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,51,0.02554155685007572,1.5234034061431885,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,52,0.024267013594508172,1.489872932434082,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,53,0.02627017389982939,1.4571149349212646,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,54,0.025388200357556342,1.5193336009979248,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,55,0.0251073032617569,1.530482530593872,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,56,0.025182886719703673,1.5103492736816406,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,57,0.025039794966578485,1.4844841957092285,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,58,0.02447968170046806,1.51027512550354,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,59,0.023596500977873802,1.496868371963501,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,60,0.0226706163585186,1.591428518295288,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,61,0.023505417928099633,1.5771687030792236,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,62,0.022282766923308373,1.5081360340118408,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,63,0.02407857611775398,1.5308072566986084,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,64,0.02591887779533863,1.5710716247558594,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,65,0.026039888933300972,1.5558679103851318,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,66,0.02432307682931423,1.5216679573059082,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,67,0.022614075914025308,1.5368421077728271,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,68,0.02234252654016018,1.5855157375335693,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,69,0.022976682856678962,1.5497357845306396,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,70,0.02195304997265339,1.5486462116241455,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,71,0.022741339877247812,1.4931304454803467,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,72,0.02258528456091881,1.5485758781433105,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,73,0.022945038266479967,1.5374603271484375,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,74,0.02365014549344778,1.5844941139221191,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,75,0.022869915515184403,1.5373375415802002,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,76,0.02326191745698452,1.5367207527160645,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,77,0.023500179499387742,1.5474178791046143,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,78,0.022383475229144096,1.5625922679901123,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,79,0.020673485547304152,1.5725734233856201,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,80,0.022309423461556434,1.5702595710754395,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,81,0.022667487934231758,1.6005971431732178,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,82,0.02201342597603798,1.6071679592132568,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,83,0.022574724406003954,1.5695550441741943,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,84,0.021212293952703475,1.56129789352417,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,85,0.021025249734520912,1.5727767944335938,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,86,0.021871483102440834,1.5381357669830322,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,87,0.02148771770298481,1.5715935230255127,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,88,0.023758746571838855,1.5584802627563477,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,89,0.021549981124699115,1.5860183238983154,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,90,0.01865873608738184,1.5749905109405518,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,91,0.020049177408218384,1.5336189270019531,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,92,0.020623030215501784,1.5211122035980225,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,93,0.02077210061252117,1.571976900100708,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,94,0.020936616510152817,1.5796802043914795,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,95,0.022118094302713873,1.5299005508422852,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,96,0.020740063451230525,1.5830152034759521,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,97,0.01954802133142948,1.5650575160980225,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,98,0.018753526844084264,1.5382802486419678,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,99,0.01928467594087124,1.61635422706604,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,100,0.020221721716225147,1.5727787017822266,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,101,0.01929360080510378,1.557955026626587,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,102,0.019791422635316847,1.5425701141357422,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,103,0.018006792366504668,1.5355961322784424,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,104,0.01752388771623373,1.6423139572143555,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,105,0.01803989376872778,1.5988991260528564,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,106,0.019292038790881635,1.6636972427368164,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,107,0.020081962868571283,1.5554664134979248,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,108,0.020052433125674726,1.5534868240356445,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,109,0.019953731037676333,1.603933572769165,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,110,0.019694149680435657,1.5636703968048096,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,111,0.019494151435792445,1.5822851657867432,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,112,0.02060811948031187,1.6193454265594482,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,113,0.019015569649636747,1.5896899700164795,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,114,0.01655249897390604,1.5861849784851074,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,115,0.018185013458132743,1.5972323417663574,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,116,0.0184220689535141,1.5713706016540527,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,117,0.0168682262301445,1.567946195602417,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,118,0.016526799872517584,1.6819300651550293,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,119,0.018931348323822022,1.6390933990478516,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,120,0.018434323370456696,1.6078271865844727,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,121,0.016666100434958933,1.5605478286743164,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,122,0.015932389162480833,1.5637314319610596,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,123,0.016874795742332936,1.5854747295379639,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,124,0.015832162387669088,1.6116700172424316,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,125,0.016968960165977477,1.6217210292816162,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,126,0.016113923005759714,1.6291730403900146,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,127,0.016370865143835546,1.5779430866241455,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,128,0.017340524382889272,1.5891809463500977,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,129,0.018038660734891892,1.573983907699585,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,130,0.017166455984115602,1.6318843364715576,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,131,0.016507445611059667,1.5992960929870605,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,132,0.015154939368367195,1.5834152698516846,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,133,0.015313753113150596,1.6159775257110596,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,134,0.015508278720080853,1.604085922241211,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,135,0.01628334168344736,1.572537899017334,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,136,0.01723056297749281,1.636871099472046,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,137,0.016221270859241486,1.643869161605835,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,138,0.01639403559267521,1.6180667877197266,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,139,0.016439789570868016,1.5545861721038818,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,140,0.01745746858417988,1.6156814098358154,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,141,0.014962810017168522,1.6638095378875732,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,142,0.015858321227133276,1.6097309589385986,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,143,0.017720992118120192,1.6446242332458496,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,144,0.017697106152772903,1.6394307613372803,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,145,0.017165479250252247,1.6477539539337158,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,146,0.015033623985946178,1.6500158309936523,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,147,0.013815610259771348,1.6373653411865234,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,148,0.01356470439583063,1.5946767330169678,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,149,0.015874837674200535,1.630296230316162,8,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,150,0.015283494219183923,1.6405420303344727,8,0.16666666666666666\n', 'NTSCC_plus_AWGN': 'model,channel,epoch,epochs_target,snr_db,train_loss,train_psnr_db,estimated_bpp,epoch_seconds,train_flops_per_step,inference_flops_per_image,batch_size,crop,images_in_epoch,amp_fp16,source_commit\nNTSCC_plus,AWGN,1,150,10.0,39792.952880859375,-2.755332043364393,12.56716638803482,31.988705158233643,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,2,150,10.0,6981.579391479492,5.323929601522509,5.221501022577286,27.936014413833618,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,3,150,10.0,2899.8765716552734,9.230145971967366,2.0459309965372086,22.147674798965454,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,4,150,10.0,2035.117774963379,10.467426791402856,1.4391547292470932,21.931376695632935,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,5,150,10.0,1534.4702911376953,11.601195251592436,1.3555161133408546,22.36415433883667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,6,150,10.0,1367.8081321716309,11.676967175571765,1.3413746878504753,21.816887855529785,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,7,150,10.0,1034.2150039672852,12.988741757761888,1.4124318361282349,22.35653328895569,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,8,150,10.0,860.473949432373,13.691158757316932,1.3835690394043922,21.719832181930542,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,9,150,10.0,790.1523857116699,14.119188622652885,1.403208665549755,21.811223030090332,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,10,150,10.0,753.8871574401855,14.208758550937098,1.3850225806236267,22.316543579101562,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,11,150,10.0,592.3934116363525,15.373168726706194,1.3910673558712006,21.595791578292847,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,12,150,10.0,594.8851890563965,15.277154761942482,1.296489842236042,21.9667387008667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,13,150,10.0,531.2410869598389,15.730459440883573,1.2792765721678734,21.560975074768066,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,14,150,10.0,542.2916355133057,15.63637102826535,1.2845947816967964,22.095397233963013,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,15,150,10.0,451.8876552581787,16.46690545853577,1.2610168308019638,21.74895143508911,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,16,150,10.0,491.3628578186035,15.877132800143066,1.2510557249188423,21.912362813949585,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,17,150,10.0,428.53265953063965,16.47285263476945,1.2621963173151016,23.17900586128235,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,18,150,10.0,442.77608489990234,16.334588785847227,1.290684811770916,22.55223536491394,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,19,150,10.0,430.241662979126,16.399056273208718,1.2986488118767738,23.259185552597046,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,20,150,10.0,462.53583908081055,15.939231115566113,1.349912278354168,22.78720736503601,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,21,150,10.0,409.3742847442627,16.54609024445658,1.4328963607549667,23.259812355041504,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,22,150,10.0,375.34513092041016,16.84652980449987,1.3376842513680458,22.74508547782898,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,23,150,10.0,336.80224800109863,17.28211327336637,1.31114911288023,23.555722951889038,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,24,150,10.0,353.61078453063965,17.003364493072716,1.321211315691471,23.02208423614502,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,25,150,10.0,365.6126117706299,16.81016237398682,1.3191985934972763,22.953298568725586,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,26,150,10.0,351.7023410797119,17.02427352948228,1.307821087539196,22.95594096183777,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,27,150,10.0,333.9455795288086,17.248168283985436,1.32345799356699,22.598791122436523,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,28,150,10.0,289.34821128845215,17.791647165157254,1.3124936148524284,23.652023792266846,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,29,150,10.0,350.45277309417725,17.004256318685318,1.3712836056947708,22.952850580215454,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,30,150,10.0,323.0689353942871,17.39843962710249,1.3645784258842468,22.527425527572632,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,31,150,10.0,332.90099239349365,17.276934549151317,1.3872633501887321,22.806499481201172,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,32,150,10.0,306.04050731658936,17.607156134288516,1.3718416839838028,22.44851326942444,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,33,150,10.0,267.94407749176025,18.145132397726066,1.3385836035013199,22.946462631225586,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,34,150,10.0,279.67390727996826,17.872319494093077,1.3454597815871239,22.650651693344116,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,35,150,10.0,298.9259195327759,17.632541580221503,1.352270171046257,22.443472623825073,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,36,150,10.0,288.36194896698,17.77113654188968,1.4059413000941277,22.69043779373169,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,37,150,10.0,275.2576484680176,18.00000496322244,1.3879760652780533,22.86424708366394,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,38,150,10.0,286.4496536254883,17.79766778517618,1.4043457582592964,23.44990634918213,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,39,150,10.0,257.75805854797363,18.27785808580352,1.3741118535399437,22.95795178413391,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,40,150,10.0,272.07842922210693,18.044610782181564,1.387479528784752,24.07218647003174,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,41,150,10.0,310.02437591552734,17.42095518306415,1.488906778395176,23.068941593170166,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,42,150,10.0,281.722225189209,17.81035442697867,1.4568607732653618,22.8352530002594,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,43,150,10.0,287.3939914703369,17.760138114826677,1.5205247327685356,23.265828371047974,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,44,150,10.0,257.0302438735962,18.23690604931171,1.4466557875275612,22.829960346221924,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,45,150,10.0,252.5364475250244,18.29284502596057,1.4601499736309052,23.61336851119995,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,46,150,10.0,249.19870567321777,18.441641474107072,1.4597879201173782,23.415640592575073,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,47,150,10.0,250.6742811203003,18.394340787942603,1.4825955629348755,23.85462760925293,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,48,150,10.0,219.5273265838623,18.95227050864638,1.4203071519732475,22.772010564804077,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,49,150,10.0,218.76111507415771,18.972722256513133,1.426989734172821,23.55838966369629,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,50,150,10.0,214.8576898574829,19.049966408820033,1.4618085250258446,22.798104524612427,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,51,150,10.0,218.01782512664795,18.980944165588458,1.4519735351204872,22.424926280975342,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,52,150,10.0,263.0146884918213,18.11001973380931,1.5207263007760048,22.731394290924072,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,53,150,10.0,238.76737117767334,18.528033344456365,1.5283488929271698,22.195539236068726,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,54,150,10.0,222.07004261016846,18.9008735595314,1.5059731230139732,22.692216396331787,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,55,150,10.0,221.9753828048706,18.913789704016907,1.468542106449604,22.078246593475342,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,56,150,10.0,201.6249485015869,19.322658963723264,1.4527445286512375,22.057023286819458,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,57,150,10.0,233.31722450256348,18.67228869039944,1.4722024202346802,22.71575093269348,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,58,150,10.0,231.58963108062744,18.723767035119167,1.501148760318756,22.49824619293213,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,59,150,10.0,217.8218593597412,18.96824797260305,1.488989733159542,22.603149890899658,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,60,150,10.0,223.33054065704346,18.835760042640356,1.498335748910904,22.036519527435303,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,61,150,10.0,210.6287441253662,19.099680811280052,1.491656482219696,22.118987798690796,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,62,150,10.0,215.98143005371094,18.936622811124767,1.5646128728985786,22.21074914932251,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,63,150,10.0,199.60976886749268,19.374471263697885,1.5360757932066917,22.41091513633728,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,64,150,10.0,205.75592613220215,19.2320840153433,1.5338219478726387,23.56390881538391,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,65,150,10.0,192.3982448577881,19.40947540349214,1.5509242564439774,22.90554928779602,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,66,150,10.0,189.7183027267456,19.5658620219209,1.5356146693229675,23.833500623703003,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,67,150,10.0,197.58380317687988,19.348914385025367,1.5937729477882385,23.347342252731323,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,68,150,10.0,194.65983772277832,19.459147478940963,1.5719042718410492,23.414296865463257,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,69,150,10.0,186.7406711578369,19.639733933372007,1.527706302702427,23.59968066215515,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,70,150,10.0,192.34382820129395,19.520486524452902,1.5252604112029076,23.50745177268982,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,71,150,10.0,212.97843265533447,19.13432602696282,1.6043367087841034,23.844562768936157,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,72,150,10.0,177.23276281356812,19.87057851026124,1.5766901224851608,23.023507595062256,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,73,150,10.0,172.67132377624512,19.959645130531122,1.5944040641188622,23.281916618347168,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,74,150,10.0,210.10797691345215,19.101827746820437,1.7186110466718674,22.717442512512207,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,75,150,10.0,171.44055461883545,20.064398955729587,1.5847230553627014,22.278446674346924,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,76,150,10.0,194.18039989471436,19.477003945139852,1.5886326357722282,22.653695821762085,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,77,150,10.0,184.8071746826172,19.693833705730906,1.610305480659008,22.199661254882812,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,78,150,10.0,176.2814598083496,19.918823940928814,1.6177062913775444,22.87313485145569,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,79,150,10.0,186.19756317138672,19.644377106933522,1.6773895025253296,22.231512308120728,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,80,150,10.0,173.04176568984985,19.989311889207496,1.6447774842381477,22.793469429016113,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,81,150,10.0,166.77951097488403,20.12078981979926,1.5978920385241508,22.20487403869629,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,82,150,10.0,177.40933513641357,19.84252565770635,1.604466587305069,22.391300439834595,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,83,150,10.0,187.56576824188232,19.60411405980297,1.6413425281643867,22.688071727752686,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,84,150,10.0,164.62313508987427,20.192889218987823,1.5782289281487465,22.156715393066406,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,85,150,10.0,165.68044185638428,20.140801077688305,1.6492278352379799,22.683557987213135,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,86,150,10.0,159.7324719429016,20.3085890566727,1.6327288299798965,22.396246671676636,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,87,150,10.0,173.96030569076538,19.90263380640708,1.6502765789628029,22.918364763259888,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,88,150,10.0,151.16767168045044,20.54210232481134,1.588710866868496,22.841444730758667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,89,150,10.0,151.34615516662598,20.521098619858083,1.5920659825205803,22.85757613182068,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,90,150,10.0,170.10539770126343,20.035856277458805,1.631488412618637,23.418977975845337,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,91,150,10.0,154.02990531921387,20.497802043107406,1.6284012347459793,22.720906734466553,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,92,150,10.0,162.1735053062439,20.24751403929048,1.6760298311710358,23.024668216705322,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,93,150,10.0,162.2310094833374,20.23353003581425,1.6726972609758377,22.581536769866943,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,94,150,10.0,176.73078203201294,19.877405983153157,1.7357294633984566,23.040013790130615,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,95,150,10.0,169.6178684234619,20.025430153572383,1.6641708463430405,22.554034948349,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,96,150,10.0,171.83164882659912,19.993847590683124,1.691560760140419,22.65778875350952,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,97,150,10.0,163.40908908843994,20.248614745781467,1.6463627070188522,23.06367564201355,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,98,150,10.0,141.36825323104858,20.82128757380783,1.6032864153385162,23.02087426185608,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,99,150,10.0,143.33438968658447,20.710414545789725,1.6520850211381912,23.720595121383667,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,100,150,10.0,140.7463879585266,20.82048858479495,1.6381006017327309,22.909340858459473,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,101,150,10.0,143.03108739852905,20.714814955791834,1.7041992172598839,23.679501056671143,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,102,150,10.0,141.67910480499268,20.75521578983202,1.6399926394224167,23.619178295135498,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,103,150,10.0,153.16044092178345,20.496929166167078,1.7224777415394783,23.475963592529297,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,104,150,10.0,139.8832974433899,20.847467490136253,1.698317289352417,23.73047399520874,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,105,150,10.0,128.3905735015869,21.229516016233173,1.6549276486039162,23.210306644439697,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,106,150,10.0,141.55389213562012,20.82284181361682,1.7054586932063103,23.738853216171265,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,107,150,10.0,151.96441221237183,20.383898278920455,1.742636226117611,22.91590404510498,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,108,150,10.0,129.4276008605957,21.25303876757065,1.654836967587471,23.591758489608765,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,109,150,10.0,153.3355450630188,20.44131366290074,1.7608824670314789,22.843791246414185,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,110,150,10.0,140.4679570198059,20.874104472960397,1.7603022307157516,22.929174423217773,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,111,150,10.0,135.67965936660767,20.97522787837366,1.738464318215847,23.576690435409546,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,112,150,10.0,144.5735263824463,20.70773414950731,1.7427054345607758,23.03432822227478,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,113,150,10.0,161.52190256118774,20.22166202396056,1.7639587074518204,23.49159336090088,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,114,150,10.0,142.37137508392334,20.76000880935886,1.775726079940796,22.95783257484436,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,115,150,10.0,128.16529870033264,21.27730841739082,1.6816172674298286,23.676251649856567,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,116,150,10.0,139.18946361541748,20.892097095879993,1.7889949977397919,22.941893100738525,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,117,150,10.0,124.44748735427856,21.34120885871544,1.7267489358782768,22.744948625564575,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,118,150,10.0,137.52435159683228,20.922531584418,1.7966705709695816,23.991820096969604,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,119,150,10.0,132.45342350006104,21.149878377122047,1.773485116660595,23.50482726097107,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,120,150,10.0,128.15565872192383,21.24763547468673,1.7483628541231155,23.660330295562744,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,121,150,10.0,139.611478805542,20.886214472872577,1.7832966819405556,23.30727458000183,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,122,150,10.0,141.2697639465332,20.80715300991916,1.7991910949349403,23.76007056236267,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,123,150,10.0,121.81045007705688,21.445807619849905,1.699307695031166,23.155982494354248,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,124,150,10.0,130.6044373512268,21.13791045250395,1.75665844976902,23.1305673122406,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,125,150,10.0,132.38040161132812,21.07369943652405,1.8003015965223312,23.510467767715454,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,126,150,10.0,135.43379402160645,21.0099706014951,1.8504435196518898,23.090473890304565,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,127,150,10.0,130.64013385772705,21.130166044810426,1.8105339854955673,23.60650634765625,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,128,150,10.0,122.01965856552124,21.424003755584643,1.7544753327965736,23.309734106063843,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,129,150,10.0,129.7198781967163,21.11750010423787,1.8228606283664703,23.44368577003479,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,130,150,10.0,134.04688262939453,21.03523985189721,1.8136849701404572,23.925583839416504,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,131,150,10.0,131.8356580734253,21.1294889386771,1.7832308188080788,23.29207420349121,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,132,150,10.0,123.72010326385498,21.41676359716802,1.7690544798970222,23.758386611938477,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,133,150,10.0,138.16042852401733,20.85922183787445,1.8665164709091187,23.0782790184021,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,134,150,10.0,130.10167837142944,21.167645666223812,1.8071175813674927,23.242480516433716,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,135,150,10.0,118.46369457244873,21.502351061408355,1.8501371815800667,22.617563724517822,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,136,150,10.0,127.02242517471313,21.238504684278368,1.8625208884477615,22.898085832595825,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,137,150,10.0,122.52700328826904,21.426381915806477,1.8137543946504593,22.369116067886353,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,138,150,10.0,133.32515954971313,21.03742954294472,1.8694257885217667,22.555596113204956,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,139,150,10.0,121.08731985092163,21.459248251382714,1.7946772053837776,22.514005422592163,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,140,150,10.0,129.2134804725647,21.24565740753367,1.9211508110165596,22.503159761428833,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,141,150,10.0,124.45774602890015,21.364747124128712,1.9548572078347206,22.938204765319824,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,142,150,10.0,122.6030535697937,21.429967929423302,1.8801392167806625,22.600131511688232,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,143,150,10.0,115.49774169921875,21.670679975345024,1.8105816021561623,22.839780807495117,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,144,150,10.0,117.85818481445312,21.585896229901714,1.8492604941129684,22.47183847427368,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,145,150,10.0,117.07393169403076,21.670140450671113,1.8435835167765617,22.528208255767822,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,146,150,10.0,116.60646772384644,21.628910944165945,1.8507669121026993,22.79869771003723,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,147,150,10.0,110.75983238220215,21.854857611144553,1.87028107047081,22.45931315422058,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,148,150,10.0,106.64383220672607,22.038316193364224,1.8445267751812935,22.939822912216187,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,149,150,10.0,115.91200542449951,21.63766895877977,1.8508959636092186,22.455626487731934,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,AWGN,150,150,10.0,122.0655198097229,21.43691310549739,1.912136696279049,22.936331033706665,225518008470,481300316170,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\n', 'NTSCC_plus_Rayleigh': 'model,channel,epoch,epochs_target,snr_db,train_loss,train_psnr_db,estimated_bpp,epoch_seconds,train_flops_per_step,inference_flops_per_image,batch_size,crop,images_in_epoch,amp_fp16,source_commit\nNTSCC_plus,Rayleigh,1,150,10.0,41349.0400390625,-2.811202739430229,13.7027148604393,22.520040035247803,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,2,150,10.0,7017.218643188477,5.06774151405387,4.9202215522527695,22.504377365112305,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,3,150,10.0,2898.397003173828,9.29714749565262,2.0190191119909286,22.898847103118896,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,4,150,10.0,2300.8088302612305,9.656817634260607,1.536063775420189,22.53314995765686,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,5,150,10.0,1812.99951171875,10.4365035897866,1.4632453247904778,23.005796194076538,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,6,150,10.0,1475.7347564697266,11.42504783771405,1.4732406809926033,22.59389042854309,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,7,150,10.0,1145.803970336914,12.360994649190873,1.4242084845900536,22.448197603225708,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,8,150,10.0,913.327766418457,13.500498150717055,1.4107868000864983,23.122345209121704,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,9,150,10.0,834.4281883239746,13.81111831148209,1.4156472608447075,22.563008308410645,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,10,150,10.0,754.2590789794922,14.216501025711354,1.40069068223238,22.98180389404297,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,11,150,10.0,669.3445224761963,14.835019753780092,1.390677236020565,22.575226545333862,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,12,150,10.0,612.1174240112305,15.17056677156123,1.293271392583847,22.992476224899292,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,13,150,10.0,673.7242393493652,14.409002396904672,1.3593964949250221,22.571376085281372,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,14,150,10.0,568.3528594970703,15.282143925593498,1.322373628616333,22.652894496917725,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,15,150,10.0,519.9338150024414,15.680981864190137,1.3550813794136047,22.98488140106201,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,16,150,10.0,488.6125316619873,15.927141627831068,1.2736799791455269,22.435651063919067,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,17,150,10.0,493.64996337890625,15.925400125935429,1.257508173584938,23.132046699523926,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,18,150,10.0,463.74486541748047,16.184196032795903,1.2852079197764397,22.70777463912964,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,19,150,10.0,436.9586353302002,16.32518959194128,1.2925826087594032,23.13384985923767,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,20,150,10.0,444.42956161499023,16.13764852189704,1.3508744165301323,22.617018938064575,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,21,150,10.0,415.8554267883301,16.388085223557514,1.3349001854658127,22.73817729949951,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,22,150,10.0,440.98152923583984,15.982039661118373,1.3497949689626694,23.092979431152344,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,23,150,10.0,377.6946964263916,16.820958819218216,1.3299430385231972,22.490659713745117,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,24,150,10.0,377.28101539611816,16.762104942288325,1.2856604009866714,23.15365719795227,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,25,150,10.0,389.902379989624,16.555149384061647,1.3185513317584991,22.51950979232788,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,26,150,10.0,382.66830253601074,16.658107930963844,1.3070334568619728,23.168643712997437,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,27,150,10.0,403.7788715362549,16.40327289575579,1.3343323469161987,22.56281805038452,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,28,150,10.0,372.47296714782715,16.656875950861078,1.4156319797039032,22.581311464309692,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,29,150,10.0,346.21431255340576,17.062190897059132,1.4035204648971558,23.132726907730103,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,30,150,10.0,344.89992904663086,17.066271554828095,1.3389126881957054,22.53792691230774,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,31,150,10.0,331.60352325439453,17.208082876044223,1.36007309705019,22.978136777877808,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,32,150,10.0,314.33510398864746,17.441877758392156,1.3455999195575714,22.577739238739014,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,33,150,10.0,284.71247005462646,17.886455640455708,1.3179657831788063,23.037822246551514,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,34,150,10.0,325.5303955078125,17.16953216237571,1.4070902317762375,22.641459941864014,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,35,150,10.0,328.9906520843506,17.267825371166225,1.4022785797715187,22.610316038131714,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,36,150,10.0,346.92087173461914,16.961591821223177,1.4301762208342552,23.13834261894226,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,37,150,10.0,305.4383201599121,17.548619724995916,1.4165218994021416,22.63246440887451,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,38,150,10.0,310.14663887023926,17.429501413080956,1.454082764685154,23.09942364692688,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,39,150,10.0,284.72668075561523,17.88446096645657,1.4294048696756363,22.69042444229126,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,40,150,10.0,301.07518196105957,17.641418381909247,1.4636427611112595,24.007716417312622,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,41,150,10.0,305.6825075149536,17.465964763475704,1.5287245362997055,23.662906885147095,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,42,150,10.0,278.29289722442627,17.900253174723723,1.4857235476374626,23.97583556175232,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,43,150,10.0,274.63812255859375,17.932486286102566,1.493614099919796,24.351113080978394,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,44,150,10.0,267.4329891204834,18.05404308061473,1.4617724642157555,23.551116466522217,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,45,150,10.0,325.36741638183594,17.227904780618747,1.4806938841938972,23.90597367286682,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,46,150,10.0,272.99556827545166,17.987910275977356,1.4714614301919937,23.138901472091675,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,47,150,10.0,261.4921474456787,18.046102900437244,1.5114870965480804,22.986191987991333,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,48,150,10.0,236.07163524627686,18.568698300579804,1.4717905521392822,23.547114610671997,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,49,150,10.0,251.1828498840332,18.24987937641717,1.5031865313649178,22.703654766082764,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,50,150,10.0,249.5067834854126,18.26558696009133,1.4935062527656555,24.371196031570435,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,51,150,10.0,262.0569877624512,18.082113963900518,1.487262561917305,23.106560707092285,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,52,150,10.0,267.0299777984619,17.98417243165192,1.5362978354096413,23.71036124229431,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,53,150,10.0,254.35997772216797,18.184364911683012,1.5332511737942696,23.592999935150146,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,54,150,10.0,225.9652156829834,18.748546326644824,1.4918580874800682,24.25970983505249,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,55,150,10.0,219.85447883605957,18.883265070593048,1.4734684079885483,24.31025457382202,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,56,150,10.0,228.6451873779297,18.5803494160967,1.4965251609683037,24.172194480895996,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,57,150,10.0,249.8910894393921,18.38297959066064,1.5353806018829346,24.150861501693726,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,58,150,10.0,226.99152946472168,18.666271856772795,1.4821219742298126,24.176467895507812,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,59,150,10.0,227.79400539398193,18.55074449669711,1.5872857794165611,24.20240879058838,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,60,150,10.0,238.77710628509521,18.44924824477762,1.7685507982969284,23.737866163253784,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,61,150,10.0,245.19473934173584,18.325651807167255,1.9795779213309288,24.429202556610107,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,62,150,10.0,250.53323650360107,18.214239872065875,1.8453193008899689,23.235646724700928,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,63,150,10.0,219.3070936203003,18.892167553290676,1.7090303376317024,23.576057195663452,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,64,150,10.0,211.44362449645996,19.089178279321043,1.6466707661747932,24.090883016586304,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,65,150,10.0,206.44568920135498,19.027847223873106,1.622465193271637,23.68265652656555,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,66,150,10.0,195.32946968078613,19.38799941453905,1.606980837881565,23.845715761184692,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,67,150,10.0,207.91192531585693,19.063302022126532,1.6272514536976814,23.5789315700531,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,68,150,10.0,198.55445194244385,19.267390522692477,1.6195780709385872,23.131535053253174,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,69,150,10.0,202.18408679962158,19.24887362928481,1.6397320106625557,23.247233867645264,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,70,150,10.0,228.55742168426514,18.639176823946986,1.6214455664157867,22.91496777534485,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,71,150,10.0,202.5975513458252,19.187888974415728,1.6013498529791832,23.368141651153564,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,72,150,10.0,183.78849601745605,19.59604157783565,1.589103676378727,22.80917453765869,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,73,150,10.0,182.43411779403687,19.663446503285154,1.619361698627472,23.32826042175293,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,74,150,10.0,212.27516651153564,18.95482872624664,1.6736816689372063,22.852824926376343,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,75,150,10.0,194.80434799194336,19.354767287816426,1.6313253343105316,22.853309392929077,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,76,150,10.0,187.6400489807129,19.47294150981587,1.6123105138540268,23.47062611579895,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,77,150,10.0,197.1684808731079,19.25924460462,1.6235572770237923,22.855468034744263,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,78,150,10.0,196.21375560760498,19.294894251718258,1.625948190689087,23.60096549987793,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,79,150,10.0,192.69738483428955,19.334900762112483,1.667699284851551,22.866288661956787,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,80,150,10.0,191.09291076660156,19.36888022043973,1.672297403216362,23.381592988967896,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,81,150,10.0,196.45355224609375,19.297133166731886,1.6853880807757378,22.7796151638031,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,82,150,10.0,185.26821327209473,19.542932578510204,1.6442170590162277,23.370110750198364,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,83,150,10.0,178.98089504241943,19.689110932086344,1.6860781013965607,22.871166229248047,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,84,150,10.0,167.5806064605713,19.974423083858703,1.6480986550450325,22.865784645080566,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,85,150,10.0,164.46697282791138,20.04796682484536,1.6459848135709763,23.367358684539795,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,86,150,10.0,169.43793869018555,19.9550010184008,1.644448071718216,22.848153352737427,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,87,150,10.0,171.09604406356812,19.936118005019612,1.7138966470956802,23.431975603103638,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,88,150,10.0,150.2711844444275,20.530665976924773,1.64288280159235,22.806061506271362,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,89,150,10.0,163.21251726150513,20.126881599039574,1.684489518404007,23.59006690979004,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,90,150,10.0,180.39251708984375,19.579387611177708,1.726236216723919,23.578407526016235,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,91,150,10.0,158.18418741226196,20.221396613911192,1.7174258455634117,23.35446572303772,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,92,150,10.0,172.891685962677,19.818228001338213,1.723371036350727,24.07010269165039,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,93,150,10.0,166.05180263519287,20.01789594871554,1.732001043856144,24.01545810699463,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,94,150,10.0,174.0342583656311,19.83910702318102,1.7641351372003555,24.1419939994812,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,95,150,10.0,167.27905797958374,19.99489153402269,1.7082859948277473,23.822604179382324,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,96,150,10.0,176.7899842262268,19.702055700244014,1.7524599134922028,23.727959632873535,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,97,150,10.0,156.2618489265442,20.26346293986687,1.701586239039898,24.790911197662354,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,98,150,10.0,161.31925344467163,20.195867437851234,1.6937437281012535,23.91642999649048,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,99,150,10.0,160.12151765823364,20.17072427959927,1.717956893146038,24.512569904327393,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,100,150,10.0,156.79281520843506,20.253307262131592,1.7448887303471565,24.146832704544067,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,101,150,10.0,153.26374340057373,20.296931506998995,1.7644638940691948,24.066624641418457,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,102,150,10.0,143.90242338180542,20.612143662859445,1.6879815012216568,24.76875114440918,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,103,150,10.0,173.11160707473755,19.85380774188596,1.7996005341410637,23.172399044036865,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,104,150,10.0,157.00701904296875,20.21528786252141,1.742096669971943,23.64422082901001,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,105,150,10.0,161.85227823257446,20.097117431876725,1.775273971259594,23.036388158798218,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,106,150,10.0,155.01375007629395,20.321662252170224,1.7341657429933548,23.619014501571655,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,107,150,10.0,155.92805528640747,20.235085086546324,1.7419947013258934,22.92130208015442,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,108,150,10.0,143.45263528823853,20.59955849898559,1.7072131261229515,23.950132369995117,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,109,150,10.0,162.60852813720703,20.065066163614794,1.8151395991444588,22.99645161628723,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,110,150,10.0,143.83264636993408,20.585233918708514,1.7419798970222473,22.88100004196167,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,111,150,10.0,163.5464949607849,20.027948546052706,1.7913448363542557,23.47038722038269,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,112,150,10.0,155.60212230682373,20.227710063101224,1.7372583746910095,23.075082302093506,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,113,150,10.0,154.15455198287964,20.286948219900097,1.7593494430184364,23.41478419303894,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,114,150,10.0,146.2304925918579,20.535696592598267,1.8013902604579926,22.942867755889893,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,115,150,10.0,139.64932918548584,20.779812072028818,1.8003129735589027,22.959189414978027,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,116,150,10.0,154.157555103302,20.31603569275451,1.8626248091459274,23.564975261688232,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,117,150,10.0,133.95377349853516,20.911204898028192,1.7704730033874512,22.919907808303833,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,118,150,10.0,150.5819444656372,20.333507702675742,1.8353237882256508,23.605074405670166,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,119,150,10.0,145.16802597045898,20.61039884706495,1.841575987637043,23.332337617874146,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,120,150,10.0,135.9186191558838,20.839691191059913,1.8492376655340195,23.71750521659851,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,121,150,10.0,139.28937196731567,20.65660622629622,1.7906989455223083,23.059417009353638,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,122,150,10.0,163.91792964935303,19.953561794081306,1.9349487647414207,23.220706939697266,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,123,150,10.0,138.60392332077026,20.765608658398715,1.82143634557724,23.528882265090942,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,124,150,10.0,133.88504028320312,20.867889361562707,1.839795008301735,22.943483352661133,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,125,150,10.0,138.42737340927124,20.735092604284393,1.8936958238482475,23.325660943984985,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,126,150,10.0,143.6514801979065,20.594762394671307,1.908023625612259,22.9375262260437,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,127,150,10.0,141.0035433769226,20.647970288188354,1.9156325161457062,23.49112606048584,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,128,150,10.0,143.24181747436523,20.628955583508795,1.8632602095603943,22.958664178848267,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,129,150,10.0,153.42424726486206,20.363721442840536,1.870009072124958,22.911504983901978,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,130,150,10.0,162.60843992233276,20.063234249712153,1.8844954147934914,23.420605897903442,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,131,150,10.0,146.35474586486816,20.54873132350144,1.9010154828429222,23.02019238471985,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,132,150,10.0,134.35641956329346,20.910001897516214,1.8779368624091148,23.42356848716736,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,133,150,10.0,138.3089485168457,20.771583320387368,1.8833899348974228,22.864949703216553,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,134,150,10.0,126.99617195129395,21.17065585983637,1.8714429661631584,22.94248104095459,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,135,150,10.0,120.32891654968262,21.323804717487064,1.8223903700709343,23.30249786376953,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,136,150,10.0,146.49135828018188,20.513073026513233,1.9494082033634186,23.075963735580444,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,137,150,10.0,137.79952096939087,20.816924791852884,1.8637294322252274,23.638261556625366,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,138,150,10.0,135.10473537445068,20.903240081781952,1.9143378362059593,23.570582628250122,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,139,150,10.0,136.50968742370605,20.84909200503125,1.9048939421772957,23.885743141174316,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,140,150,10.0,143.03124856948853,20.512204840308694,1.9223804846405983,23.614594221115112,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,141,150,10.0,126.23761177062988,21.164619427343553,1.9240339547395706,24.37408971786499,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,142,150,10.0,131.61556482315063,20.994203899306484,1.9329662695527077,24.008030652999878,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,143,150,10.0,128.6069564819336,21.072502662627382,1.8831132799386978,23.878161430358887,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,144,150,10.0,127.77489233016968,21.049748298532414,1.8845093473792076,23.60043454170227,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,145,150,10.0,132.05419874191284,20.974157946849317,1.9183058738708496,24.060317039489746,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,146,150,10.0,121.14570236206055,21.322975073263635,1.9214284345507622,24.627951860427856,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,147,150,10.0,127.45696353912354,21.093909485954157,1.974757820367813,23.712194442749023,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,148,150,10.0,121.36186170578003,21.27488364495281,1.909542664885521,24.261477947235107,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,149,150,10.0,127.34014701843262,21.162953981265964,1.8894806876778603,23.461796522140503,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\nNTSCC_plus,Rayleigh,150,150,10.0,124.92308855056763,21.20550629251973,1.937358982861042,23.61973810195923,225518090394,481300316174,8,256,128,True,765c6f6c11cf3bbe249a26b624dbaf47a4f0676d\n'}
epochs = pd.concat([pd.read_csv(io.StringIO(txt)).assign(run=name) for name, txt in EPOCH_CSVS.items()], ignore_index=True)
# Plot whichever recorded loss column each run provides.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, frame in epochs.groupby("run"):
    col = "train_mse" if "train_mse" in frame else "train_loss" if "train_loss" in frame else "train_loss_mse_255"
    if col in frame: axes[0].plot(frame["epoch"], pd.to_numeric(frame[col], errors="coerce"), label=name)
    if "epoch_seconds" in frame: axes[1].plot(frame["epoch"], pd.to_numeric(frame["epoch_seconds"], errors="coerce"), label=name)
axes[0].set_title("Recorded training loss"); axes[1].set_title("Seconds per epoch")
for ax in axes: ax.set_xlabel("Epoch"); ax.grid(alpha=.25); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()
display(Markdown("ADJSCC also completed 150 epochs per channel. Its epoch CSVs remain in the linked ADJSCC Kaggle run; the aggregate is included above. The local package did not include those per-epoch rows when this notebook was assembled."))


In [ ]:
# Per-image Kodak data included in the local run artifacts.
PER_IMAGE_CSVS = {'DeepJSCC': 'model,channel,snr_db,image,psnr_db,ssim,msssim,real_symbols,complex_channel_uses,cbr\nDeepJSCC,AWGN,10.0,kodim01,21.43340263699568,0.4605445861816406,0.8283419609069824,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim02,23.698968239040124,0.5910143256187439,0.8343927264213562,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim03,24.006610609358603,0.7504670023918152,0.8772806525230408,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim04,23.17017546522488,0.6792846918106079,0.8674923181533813,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim05,20.611312598525608,0.5202287435531616,0.847913920879364,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim06,22.610474433856425,0.5678064227104187,0.8470005989074707,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim07,24.95639874355151,0.7681567072868347,0.9057396650314331,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim08,19.094292244687423,0.5025618076324463,0.8593036532402039,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim09,25.255051270718774,0.7483531832695007,0.8996027708053589,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim10,25.576342639701007,0.7348123788833618,0.8993462920188904,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim11,23.720570944911422,0.6219055652618408,0.8728143572807312,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim12,24.243731909690155,0.7341151237487793,0.8897705078125,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim13,19.882630565919644,0.4077429175376892,0.8154768943786621,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim14,22.651857278064874,0.5776296854019165,0.8569671511650085,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim15,23.569709371209644,0.7240301370620728,0.9037925004959106,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim16,26.50940003635911,0.6604882478713989,0.8779044151306152,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim17,26.42234576410829,0.751697301864624,0.9265649318695068,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim18,22.87966554918303,0.5772435665130615,0.8566379547119141,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim19,23.170345001186558,0.6355150938034058,0.8674356341362,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim20,24.810330186369953,0.778167724609375,0.9320201873779297,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim21,23.614634585756782,0.6710586547851562,0.8848098516464233,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim22,24.05299357036619,0.6279572248458862,0.849965512752533,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim23,22.706884777637597,0.7915891408920288,0.8722074031829834,393216,196608,0.16666666666666666\nDeepJSCC,AWGN,10.0,kodim24,22.19121671155452,0.5996863842010498,0.8735150694847107,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim01,19.860650619890706,0.3149241805076599,0.6802804470062256,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim02,19.508728632289046,0.5040397047996521,0.771343469619751,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim03,20.612716152521692,0.6964635252952576,0.776839017868042,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim04,20.202432088537805,0.6260297894477844,0.7972409129142761,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim05,18.181987429131688,0.35064029693603516,0.6647676825523376,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim06,19.820931482088678,0.46234267950057983,0.682163417339325,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim07,21.91190658404831,0.6134834289550781,0.8023084402084351,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim08,16.524364481440738,0.3449789881706238,0.690968930721283,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim09,20.732436018007245,0.6797795295715332,0.8400163650512695,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim10,23.00459230971914,0.6538388133049011,0.7631387710571289,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim11,21.733379818291905,0.5265095233917236,0.7374655604362488,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim12,20.402358157128553,0.6494988203048706,0.7035094499588013,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim13,18.256312059064484,0.2705322504043579,0.6281886100769043,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim14,20.368171323170834,0.4418865144252777,0.7307552099227905,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim15,19.597193728113695,0.5985109210014343,0.8388852477073669,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim16,20.66082663783775,0.5339365005493164,0.5478882789611816,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim17,22.63452901402316,0.505901575088501,0.8134534358978271,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim18,20.231054653580124,0.4334275424480438,0.6277710795402527,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim19,21.29343911354949,0.5581701993942261,0.798837423324585,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim20,21.606598217276407,0.7075926661491394,0.8772899508476257,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim21,20.149226063352895,0.5768390893936157,0.79123854637146,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim22,20.753081270926277,0.5374479293823242,0.7205602526664734,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim23,20.255840262506595,0.7095434069633484,0.8207847476005554,393216,196608,0.16666666666666666\nDeepJSCC,Rayleigh,10.0,kodim24,19.66194216272957,0.48059624433517456,0.764950156211853,393216,196608,0.16666666666666666\n', 'NTSCC_plus_AWGN': 'model,channel,snr_db,image,psnr_db,ssim,msssim,complex_channel_uses,cbr\nNTSCC_plus,AWGN,10.0,kodim01,21.402482986450195,0.46129482984542847,0.8203120231628418,80698.0,0.0684085422092014\nNTSCC_plus,AWGN,10.0,kodim02,20.54510498046875,0.49493059515953064,0.7243842482566833,45107.0,0.038237677680121526\nNTSCC_plus,AWGN,10.0,kodim03,17.868877410888672,0.4526536166667938,0.6827144622802734,36578.0,0.031007554796006944\nNTSCC_plus,AWGN,10.0,kodim04,20.018497467041016,0.5439622402191162,0.7269507646560669,43981.0,0.0372831556532118\nNTSCC_plus,AWGN,10.0,kodim05,20.88838768005371,0.5199162364006042,0.8467792272567749,87664.0,0.0743136935763889\nNTSCC_plus,AWGN,10.0,kodim06,22.471277236938477,0.5296902656555176,0.7983217239379883,60695.0,0.051451789008246526\nNTSCC_plus,AWGN,10.0,kodim07,23.2613525390625,0.6952646970748901,0.8366225957870483,52923.0,0.044863382975260414\nNTSCC_plus,AWGN,10.0,kodim08,19.492996215820312,0.5132796764373779,0.8578262329101562,91673.0,0.07771216498480903\nNTSCC_plus,AWGN,10.0,kodim09,22.067066192626953,0.679559588432312,0.8323910236358643,42971.0,0.03642696804470486\nNTSCC_plus,AWGN,10.0,kodim10,21.85690689086914,0.6147552728652954,0.7442799806594849,41717.0,0.0353639390733507\nNTSCC_plus,AWGN,10.0,kodim11,23.160888671875,0.5390491485595703,0.8253681659698486,55070.0,0.046683417426215276\nNTSCC_plus,AWGN,10.0,kodim12,22.814178466796875,0.6597297191619873,0.7922646403312683,36815.0,0.031208462185329862\nNTSCC_plus,AWGN,10.0,kodim13,20.12094497680664,0.3981275260448456,0.8059553503990173,84061.0,0.07125939263237847\nNTSCC_plus,AWGN,10.0,kodim14,22.51336097717285,0.5159306526184082,0.8278703689575195,68223.0,0.057833353678385414\nNTSCC_plus,AWGN,10.0,kodim15,19.49456214904785,0.5151889324188232,0.7880266904830933,44266.0,0.03752475314670139\nNTSCC_plus,AWGN,10.0,kodim16,22.62181282043457,0.5255637168884277,0.7536894083023071,43274.0,0.03668382432725695\nNTSCC_plus,AWGN,10.0,kodim17,22.66864013671875,0.5064125657081604,0.8299896121025085,45634.0,0.03868442111545139\nNTSCC_plus,AWGN,10.0,kodim18,20.902673721313477,0.49134212732315063,0.7767530679702759,65343.0,0.055391947428385414\nNTSCC_plus,AWGN,10.0,kodim19,19.633699417114258,0.4710562229156494,0.7080540657043457,52566.0,0.044560750325520836\nNTSCC_plus,AWGN,10.0,kodim20,19.591562271118164,0.6674091219902039,0.7986965179443359,41471.0,0.035155402289496526\nNTSCC_plus,AWGN,10.0,kodim21,20.8457088470459,0.5996946692466736,0.804611325263977,52223.0,0.04426998562282986\nNTSCC_plus,AWGN,10.0,kodim22,23.330095291137695,0.5710868835449219,0.7982997298240662,52982.0,0.044913397894965276\nNTSCC_plus,AWGN,10.0,kodim23,18.109561920166016,0.5803844332695007,0.749796986579895,38635.0,0.0327512953016493\nNTSCC_plus,AWGN,10.0,kodim24,21.363323211669922,0.5445016622543335,0.8232321739196777,66046.0,0.05598788791232639\n', 'NTSCC_plus_Rayleigh': 'model,channel,snr_db,image,psnr_db,ssim,msssim,complex_channel_uses,cbr\nNTSCC_plus,Rayleigh,10.0,kodim01,21.58173370361328,0.47184568643569946,0.8236074447631836,84102.0,0.07129414876302083\nNTSCC_plus,Rayleigh,10.0,kodim02,19.7816104888916,0.60387122631073,0.7401499152183533,44911.0,0.038071526421440974\nNTSCC_plus,Rayleigh,10.0,kodim03,17.923892974853516,0.5500379800796509,0.7220334410667419,37568.0,0.03184678819444445\nNTSCC_plus,Rayleigh,10.0,kodim04,22.466270446777344,0.6215094923973083,0.777235746383667,45461.0,0.0385377671983507\nNTSCC_plus,Rayleigh,10.0,kodim05,20.554140090942383,0.5000437498092651,0.8260678052902222,90529.0,0.0767423841688368\nNTSCC_plus,Rayleigh,10.0,kodim06,21.725475311279297,0.52309650182724,0.7716394662857056,63514.0,0.053841484917534724\nNTSCC_plus,Rayleigh,10.0,kodim07,24.273149490356445,0.7163235545158386,0.8531791567802429,55608.0,0.047139485677083336\nNTSCC_plus,Rayleigh,10.0,kodim08,19.416378021240234,0.5127749443054199,0.859288215637207,95472.0,0.0809326171875\nNTSCC_plus,Rayleigh,10.0,kodim09,23.22933578491211,0.6927449703216553,0.8550904989242554,44007.0,0.037305196126302086\nNTSCC_plus,Rayleigh,10.0,kodim10,22.479373931884766,0.6531922221183777,0.7774786949157715,43989.0,0.037289937337239586\nNTSCC_plus,Rayleigh,10.0,kodim11,23.435213088989258,0.5615600943565369,0.8362829685211182,57372.0,0.048634847005208336\nNTSCC_plus,Rayleigh,10.0,kodim12,22.83748435974121,0.6767486929893494,0.8081083297729492,38372.0,0.03252834743923611\nNTSCC_plus,Rayleigh,10.0,kodim13,20.0502986907959,0.3959733843803406,0.8032419681549072,87413.0,0.07410091824001735\nNTSCC_plus,Rayleigh,10.0,kodim14,22.673999786376953,0.5189183950424194,0.8265168070793152,71043.0,0.060223897298177086\nNTSCC_plus,Rayleigh,10.0,kodim15,18.024412155151367,0.5593299865722656,0.79038405418396,45178.0,0.03829786512586805\nNTSCC_plus,Rayleigh,10.0,kodim16,23.033710479736328,0.576822817325592,0.7975528240203857,45173.0,0.0382936265733507\nNTSCC_plus,Rayleigh,10.0,kodim17,22.768169403076172,0.5318491458892822,0.8255904912948608,48149.0,0.040816413031684026\nNTSCC_plus,Rayleigh,10.0,kodim18,21.339252471923828,0.49881577491760254,0.7868871688842773,67675.0,0.05736880832248264\nNTSCC_plus,Rayleigh,10.0,kodim19,19.96728515625,0.5481588244438171,0.773443341255188,54749.0,0.0464113023546007\nNTSCC_plus,Rayleigh,10.0,kodim20,14.534048080444336,0.656623363494873,0.7540795803070068,42977.0,0.0364320543077257\nNTSCC_plus,Rayleigh,10.0,kodim21,19.493276596069336,0.556827187538147,0.6827530264854431,54227.0,0.0459687974717882\nNTSCC_plus,Rayleigh,10.0,kodim22,23.88559341430664,0.5845715403556824,0.8060671091079712,55542.0,0.047083536783854164\nNTSCC_plus,Rayleigh,10.0,kodim23,17.699512481689453,0.6184325218200684,0.7684034109115601,38599.0,0.0327207777235243\nNTSCC_plus,Rayleigh,10.0,kodim24,20.903820037841797,0.5514144897460938,0.8298238515853882,68906.0,0.05841233995225695\n'}
per_image = pd.concat([pd.read_csv(io.StringIO(txt)).assign(run=name) for name, txt in PER_IMAGE_CSVS.items()], ignore_index=True)
display(Markdown("## Per-image Kodak records available in this notebook"))
display(per_image.head(12))
print("Embedded Kodak per-image records:", len(per_image), "(DeepJSCC and NTSCC_plus). ADJSCC per-image CSVs are linked from its Kaggle output page.")


In [ ]:
# Export reproducible report artifacts into the Kaggle run output directory.
from pathlib import Path
work = Path("/kaggle/working/consolidated_jscc_results"); work.mkdir(parents=True, exist_ok=True)
summary.to_csv(work/"three_model_kodak_summary.csv", index=False)
detail.to_csv(work/"three_model_complete_comparison.csv", index=False)
epochs.to_csv(work/"available_epoch_histories.csv", index=False)
per_image.to_csv(work/"available_per_image_kodak.csv", index=False)
print("Wrote report tables to", work)
print("Training runs (private Kaggle kernels):")
for u in [
 "https://www.kaggle.com/code/hmmkk20/deepjscc-div2k-kodak-150ep",
 "https://www.kaggle.com/code/hmmkk20/adjscc-div2k-kodak-150ep-20261016",
 "https://www.kaggle.com/code/hmmkk20/ntscc-plus-div2k-kodak-150ep-20261016",
]: print(u)
